## Step 3: ディリクレ多項分布モデル

### モデルの導出

カテゴリの購入回数 $n_i$ が与えられたとき、消費者 $i$ が各ブランド $j = 1, \ldots, J$ を選択する回数 $\mathbf{x}_i = (x_{i1}, \ldots, x_{iJ})$ をモデリングする。

**仮定1（個人レベル）**: 各消費者はブランド選択確率ベクトル $\boldsymbol{\theta}_i = (\theta_{i1}, \ldots, \theta_{iJ})$ を持ち、各購買機会で独立にブランドを選択する。

$$
\mathbf{x}_i \mid \boldsymbol{\theta}_i, n_i \sim \mathrm{Multinomial}(n_i, \boldsymbol{\theta}_i)
$$

**仮定2（集団レベル）**: 消費者間のブランド選好の異質性をディリクレ分布で表現する。

$$
\boldsymbol{\theta}_i \sim \mathrm{Dirichlet}(\boldsymbol{\alpha}), \quad \boldsymbol{\alpha} = (\alpha_1, \ldots, \alpha_J)
$$

ここで $\alpha_j > 0$ はブランド $j$ への選好の強さを表す。

**導出**: $\boldsymbol{\theta}$ を積分消去すると、ディリクレ多項分布（複合多項分布）が得られる。

$$
P(\mathbf{x}_i \mid n_i, \boldsymbol{\alpha}) = \binom{n_i}{x_{i1}, \ldots, x_{iJ}} \frac{B(\mathbf{x}_i + \boldsymbol{\alpha})}{B(\boldsymbol{\alpha})}
$$

ここで $B(\boldsymbol{\alpha}) = \frac{\prod_{j=1}^{J} \Gamma(\alpha_j)}{\Gamma(\sum_{j=1}^{J} \alpha_j)}$ は多変量ベータ関数である。

### パラメータの意味

| パラメータ | 意味 |
|-----------|------|
| $\alpha_j$ | ブランド $j$ への選好パラメータ。大きいほどそのブランドが好まれる |
| $S = \sum_j \alpha_j$ | 集中度パラメータ。大きいほどブランドロイヤルティが高い（スイッチングが少ない） |
| $\alpha_j / S$ | ブランド $j$ の期待シェア |

### 3.1 MLE によるパラメータ推定

In [ ]:
from scipy.special import gammaln

brand_cols = sorted(df_hh_brand_wide.columns.tolist())
X_brand = df_hh_brand_wide[brand_cols].values  # (n_households, J)
n_occasions = X_brand.sum(axis=1)              # total occasions per household
J = X_brand.shape[1]

def dirichlet_multinomial_loglik(log_alpha, X, n):
    """Log-likelihood of Dirichlet-Multinomial distribution."""
    alpha = np.exp(log_alpha)  # ensure positivity
    S = alpha.sum()
    ll = 0.0
    ll += gammaln(n + 1).sum() - gammaln(X + 1).sum()  # multinomial coefficient
    ll += gammaln(S) - gammaln(n + S).sum()              # B(alpha) denominator
    ll += (gammaln(X + alpha) - gammaln(alpha)).sum()    # B(x + alpha) numerator
    return ll

def dm_neg_loglik(log_alpha, X, n):
    return -dirichlet_multinomial_loglik(log_alpha, X, n)

# Initial: proportional to observed shares
share_init = X_brand.sum(axis=0) / X_brand.sum()
alpha_init = share_init * J  # scale by number of brands

result_dm = minimize(
    dm_neg_loglik,
    x0=np.log(alpha_init),
    args=(X_brand, n_occasions),
    method="Nelder-Mead",
    options={"maxiter": 50000, "xatol": 1e-8, "fatol": 1e-8},
)

alpha_hat_dm = np.exp(result_dm.x)
S_hat = alpha_hat_dm.sum()

print("=== Dirichlet-Multinomial MLE ===")
print(f"Converged: {result_dm.success}")
for j, col in enumerate(brand_cols):
    print(f"  α_{col}: {alpha_hat_dm[j]:.4f}  (share: {alpha_hat_dm[j]/S_hat:.1%})")
print(f"  S (sum of α): {S_hat:.4f}")

### 3.2 実測ブランドシェア vs フィッティング結果

In [ ]:
observed_share = X_brand.sum(axis=0) / X_brand.sum()
model_share = alpha_hat_dm / S_hat

df_share_comparison = pd.DataFrame({
    "Brand": brand_cols,
    "Observed Share": observed_share,
    "Model Share": model_share,
})

fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(len(brand_cols))
bar_width = 0.35
ax.bar(x_pos - bar_width / 2, observed_share, bar_width, label="Observed", alpha=0.7)
ax.bar(x_pos + bar_width / 2, model_share, bar_width, label="Dirichlet-Multinomial", alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(brand_cols, rotation=30, ha="right")
ax.set_ylabel("Share")
ax.set_title("Brand Share: Observed vs Dirichlet-Multinomial")
ax.legend()

for i, (o, m) in enumerate(zip(observed_share, model_share)):
    ax.text(i - bar_width / 2, o + 0.005, f"{o:.1%}", ha="center", fontsize=8)
    ax.text(i + bar_width / 2, m + 0.005, f"{m:.1%}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

### 3.3 パラメータの解釈（ブランドロイヤルティとスイッチング）

In [ ]:
print("=== Dirichlet-Multinomial Interpretation ===\n")

# S parameter interpretation
print(f"S (concentration) = {S_hat:.4f}")
if S_hat < 1:
    print("  -> S < 1: Very low loyalty, heavy brand switching")
elif S_hat < 5:
    print("  -> S is small: Moderate switching tendency")
else:
    print("  -> S is large: High brand loyalty")

# Expected repeat-buying proportion for each brand
# P(choose brand j again | chose j last time) = (alpha_j + 1) / (S + 1)
print(f"\nRepeat-buying probability (given last purchase was brand j):")
for j, col in enumerate(brand_cols):
    repeat_prob = (alpha_hat_dm[j] + 1) / (S_hat + 1)
    print(f"  {col}: {repeat_prob:.1%}")

# Switching matrix: P(choose brand k | chose brand j) = alpha_k / (S + 1) for k != j
print(f"\nSwitching matrix (row = last brand, col = next brand):")
switch_matrix = np.zeros((J, J))
for j in range(J):
    for k in range(J):
        if j == k:
            switch_matrix[j, k] = (alpha_hat_dm[k] + 1) / (S_hat + 1)
        else:
            switch_matrix[j, k] = alpha_hat_dm[k] / (S_hat + 1)

df_switch = pd.DataFrame(switch_matrix, index=brand_cols, columns=brand_cols)
df_switch.round(3)